# Pore Size Distribution

This function follows these steps:
1. **DGrid Generation**: If a precomputed DGrid is not provided, a new DGrid is generated from the atomic structure using the specified spacing.
2. **Cell Matrix Completion**: The cell matrix is completed from the atomic structure to ensure proper periodic boundary conditions.
3. **Voronoi Tessellation**: A box is created from the cell matrix, and the Voronoi tessellation is computed based on the atomic positions.
4. **Fractional Coordinates Extraction**: The fractional coordinates of the Voronoi nodes are extracted from the tessellation results.
5. **DGrid Interpolation**: An interpolator for the DGrid is created to facilitate the calculation of node radii.
6. **Node Radii Calculation**: The radii of the Voronoi nodes are calculated using the interpolator.
7. **Random Sampling Points Generation**: A specified number of random sampling points are generated in fractional coordinates.
8. **Accessibility Filtering**: The accessible sampling points are filtered based on the DGrid interpolation results.
9. **Sample Point Radii Calculation**: The radii for the accessible sampling points are obtained using the DGrid interpolator.
10. **Distance Calculation**: For each block of sample points, the distances to the Voronoi nodes are calculated, and valid radii are determined.
11. **Result Compilation**: The maximum valid radii and the sample block radii are combined to produce the final pore sizes.
12. **Dask Processing**: The calculations are parallelized using Dask to handle large datasets efficiently.


## Make imports

In [1]:
from ase.io import read
from pathlib import Path 
import numpy as np
import mofography as mgr


### Pick a random CIF	
(change paths here as needed )

In [29]:
path_to_mgr = Path('C:/Users/arung/OneDrive - The University of Manchester/my_cloned_repos/22sep2024/mofography/')

list_of_cifs= list((path_to_mgr / 'examples' / 'TestCIFs' ).rglob('*.cif'))
path_to_cif = np.random.choice(list_of_cifs)
print('The selected CIF is:\n', path_to_cif.name)

The selected CIF is:
 ABAVIJ_clean.cif


## Define functions

### Compute pore sizes for psd

In [30]:

def compute_pore_sizes_for_psd(atoms, dgrid=None,  num_samples=50000, spacing=0.25, blocksize=500):
	"""
	Computes the pore sizes from an atomic structure using Voronoi tessellation and DGrid interpolation, to be histogrammed to get the Pore Size Distribution (PSD).

	This function follows these steps:
	1. **DGrid Generation**: If a precomputed DGrid is not provided, a new DGrid is generated from the atomic structure using the specified spacing.
	2. **Cell Matrix Completion**: The cell matrix is completed from the atomic structure to ensure proper periodic boundary conditions.
	3. **Voronoi Tessellation**: A box is created from the cell matrix, and the Voronoi tessellation is computed based on the atomic positions.
	4. **Fractional Coordinates Extraction**: The fractional coordinates of the Voronoi nodes are extracted from the tessellation results.
	5. **DGrid Interpolation**: An interpolator for the DGrid is created to facilitate the calculation of node radii.
	6. **Node Radii Calculation**: The radii of the Voronoi nodes are calculated using the interpolator.
	7. **Random Sampling Points Generation**: A specified number of random sampling points are generated in fractional coordinates.
	8. **Accessibility Filtering**: The accessible sampling points are filtered based on the DGrid interpolation results.
	9. **Sample Point Radii Calculation**: The radii for the accessible sampling points are obtained using the DGrid interpolator.
	10. **Distance Calculation**: For each block of sample points, the distances to the Voronoi nodes are calculated, and valid radii are determined.
	11. **Result Compilation**: The maximum valid radii and the sample block radii are combined to produce the final pore sizes.
	12. **Dask Processing**: The calculations are parallelized using Dask to handle large datasets efficiently.

	Parameters:
	- atoms (Atoms): The ASE Atoms object containing atomic structure data.
	- dgrid (DGrid, optional): Precomputed DGrid. If None, it will be computed using the default spacing.
	- num_samples (int): The number of random sampling points to generate. Default is 50000.
	- spacing (float): The spacing for the DGrid computation. Default is 0.25.
	- blocksize (int): The size of the blocks for Dask array processing. Default is 1000. If you have less RAM, and is 
	getting a seg fault as a result, please reduce this number or the num_samples.

	Returns:
	- pore_sizes (ndarray): An array of computed pore sizes.
	"""
	def print_stat(data):
		"""Compute and print basic statistics of a numpy array."""
		max_value = np.max(data)
		min_value = np.min(data)
		mean_value = np.mean(data)
		variance_value = np.var(data)
		# print(f"Name of variable: )
		print(f"shape: {data.shape}")
		
		print(f"Max: {max_value}")
		print(f"Min: {min_value}")
		print(f"Mean: {mean_value}")
		print(f"Variance: {variance_value}")


	import numpy as np 
	import dask.array as da
	from ase.geometry import complete_cell
	from freud.locality import Voronoi 
	from freud.box import Box 
	import mofography as mgr
	# Generate a DGrid from the atomic structure if not provided
	if dgrid is None:
		dgrid = mgr.dgrid_from_atoms_cpu(atoms, spacing=spacing)

	# Complete the cell matrix from the atomic structure
	cell = complete_cell(atoms.get_cell().T)

	# Create a box for the Voronoi tessellation
	box = Box.from_matrix(cell)

	# Initialize and compute the Voronoi tessellation
	voro = Voronoi()
	voro.compute((box, atoms.get_positions()))

	# Get the fractional coordinates of the Voronoi nodes
	nodes_fractional = mgr.get_fractional_coordinates(np.vstack(voro.polytopes), atoms)

	# Create an interpolator for the DGrid
	dgrid_interpolator = mgr.interpolate_dgrid(dgrid)

	# Calculate the node radii using the interpolator
	node_radii = dgrid_interpolator(nodes_fractional % 1)

	print('==================================================')
	# Print statistics of the node radii
	print('Statistics on sphere sizes at Voronoi nodes \n(-ve values mean inside framework atoms)')
	print_stat(node_radii)

	print('==================================================')
	# Generate random sampling points in fractional coordinates
	sample_points_fractional = np.random.random((num_samples, 3))
	print('Statistics on sphere sizes at all sampling points \n(-ve values mean inside framework atoms)')
	print_stat(dgrid_interpolator(sample_points_fractional%1))
	print('==================================================')
	# Filter accessible sampling points based on DGrid interpolation
	sample_points_accessible = sample_points_fractional[dgrid_interpolator(sample_points_fractional) > 0]

	# Get the radii for the accessible sampling points
	sample_point_radii = dgrid_interpolator(sample_points_accessible)

	# Print statistics of the sample point radii
	print('Statistics on sphere sizes at sampling points in the void space')
	print_stat(sample_point_radii)

	print('==================================================')
	def points_to_nodes_distances(sample_points_block, nodes_fractional, cell=cell):
		"""
		Calculate distances from sample points to Voronoi nodes and determine valid radii.
		This function will be applied to each block of sample points, to be parallelized using Dask.
		
		Parameters:
		- sample_points_block (ndarray): Block of sample points in Cartesian coordinates.
		- nodes_fractional (ndarray): Fractional coordinates of the Voronoi nodes.
		- cell (ndarray): The cell matrix for coordinate transformation.

		Returns:
		- result (ndarray): An array of valid radii for the sample points.
		"""
		# Calculate the difference between sample points and nodes
		diff = np.expand_dims(sample_points_block, axis=1) - np.expand_dims(nodes_fractional, axis=0)
		diff = (diff - np.round(diff)).reshape(-1, 3)  # Wrap around periodic boundaries
		diff = np.dot(cell, diff.T).T  # Transform to Cartesian coordinates
		diff = np.linalg.norm(diff, axis=1).reshape(-1, nodes_fractional.shape[0])  # Compute distances

		# Print statistics of the distances
		# print_stat(diff)

		# Create a mask for valid node radii
		mask = node_radii > diff
		valid_radii = np.where(mask, node_radii, -np.inf)  # Replace invalid radii with -inf
		max_valid_radii = np.max(valid_radii, axis=1)  # Get the maximum valid radii

		# Print statistics of the maximum valid radii
		# print_stat(max_valid_radii)

		# Get the radii for the sample block
		sample_block_radii = dgrid_interpolator(sample_points_block)

		# Combine sample block radii and maximum valid radii
		result = np.array([s if np.logical_and(m != -np.inf, s > m) else m for s, m in zip(sample_block_radii, max_valid_radii)])

		return result * 2  # Return the result scaled by a factor of 2 converting radii to diameters

	# Create a Dask array from the accessible sample points
	sample_points_da = da.from_array(sample_points_accessible, chunks=(blocksize, 3))

	# Map the distance calculation function over the Dask array
	pore_sizes = sample_points_da.map_blocks(
		points_to_nodes_distances,
		nodes_fractional=nodes_fractional,
		cell=cell,
		chunks=(blocksize,),
		dtype='float32',
		drop_axis=1
	).compute()  # Compute the result
	print('For trustworthy sampled points')
	# Print statistics of the computed pore sizes
	print_stat(pore_sizes[np.isfinite(pore_sizes)])  # Only print finite values
	print('==================================================')
	return pore_sizes  # Return the computed pore sizes

In [36]:

def compute_pore_sizes_for_psd(atoms, dgrid=None,  num_samples=50000, spacing=0.25, blocksize=500, probe_radius=0.0):
	"""
	Computes the pore sizes from an atomic structure using Voronoi tessellation and DGrid interpolation, to be histogrammed to get the Pore Size Distribution (PSD).

	This function follows these steps:
	1. **DGrid Generation**: If a precomputed DGrid is not provided, a new DGrid is generated from the atomic structure using the specified spacing.
	2. **Cell Matrix Completion**: The cell matrix is completed from the atomic structure to ensure proper periodic boundary conditions.
	3. **Voronoi Tessellation**: A box is created from the cell matrix, and the Voronoi tessellation is computed based on the atomic positions.
	4. **Fractional Coordinates Extraction**: The fractional coordinates of the Voronoi nodes are extracted from the tessellation results.
	5. **DGrid Interpolation**: An interpolator for the DGrid is created to facilitate the calculation of node radii.
	6. **Node Radii Calculation**: The radii of the Voronoi nodes are calculated using the interpolator.
	7. **Random Sampling Points Generation**: A specified number of random sampling points are generated in fractional coordinates.
	8. **Accessibility Filtering**: The accessible sampling points are filtered based on the DGrid interpolation results.
	9. **Sample Point Radii Calculation**: The radii for the accessible sampling points are obtained using the DGrid interpolator.
	10. **Distance Calculation**: For each block of sample points, the distances to the Voronoi nodes are calculated, and valid radii are determined.
	11. **Result Compilation**: The maximum valid radii and the sample block radii are combined to produce the final pore sizes.
	12. **Dask Processing**: The calculations are parallelized using Dask to handle large datasets efficiently.

	Parameters:
	- atoms (Atoms): The ASE Atoms object containing atomic structure data.
	- dgrid (DGrid, optional): Precomputed DGrid. If None, it will be computed using the default spacing.
	- num_samples (int): The number of random sampling points to generate. Default is 50000.
	- spacing (float): The spacing for the DGrid computation. Default is 0.25.
	- blocksize (int): The size of the blocks for Dask array processing. Default is 1000. If you have less RAM, and is 
	getting a seg fault as a result, please reduce this number or the num_samples.

	Returns:
	- pore_sizes (ndarray): An array of computed pore sizes.
	"""
	def print_stat(data):
		"""Compute and print basic statistics of a numpy array."""
		max_value = np.max(data)
		min_value = np.min(data)
		mean_value = np.mean(data)
		variance_value = np.var(data)
		# print(f"Name of variable: )
		print(f"shape: {data.shape}")
		
		print(f"Max: {max_value}")
		print(f"Min: {min_value}")
		print(f"Mean: {mean_value}")
		print(f"Variance: {variance_value}")


	import numpy as np 
	import dask.array as da
	from ase.geometry import complete_cell
	from freud.locality import Voronoi 
	from freud.box import Box 
	import mofography as mgr
	# Generate a DGrid from the atomic structure if not provided
	if dgrid is None:
		dgrid = mgr.dgrid_from_atoms_cpu(atoms, spacing=spacing)

	# Complete the cell matrix from the atomic structure
	cell = complete_cell(atoms.get_cell().T)

	# Create a box for the Voronoi tessellation
	box = Box.from_matrix(cell)

	# Initialize and compute the Voronoi tessellation
	voro = Voronoi()
	voro.compute((box, atoms.get_positions()))

	# Get the fractional coordinates of the Voronoi nodes
	nodes_fractional = mgr.get_fractional_coordinates(np.vstack(voro.polytopes), atoms)

	# Create an interpolator for the DGrid
	dgrid_interpolator = mgr.interpolate_dgrid(dgrid)

	# Calculate the node radii using the interpolator
	node_radii = dgrid_interpolator(nodes_fractional % 1)

	print('==================================================')
	# Print statistics of the node radii
	print('Statistics on sphere sizes at Voronoi nodes \n(-ve values mean inside framework atoms)')
	print_stat(node_radii)

	print('==================================================')
	# Generate random sampling points in fractional coordinates
	sample_points_fractional = np.random.random((num_samples, 3))
	print('Statistics on sphere sizes at all sampling points \n(-ve values mean inside framework atoms)')
	print_stat(dgrid_interpolator(sample_points_fractional%1))
	print('==================================================')
	# Filter accessible sampling points based on DGrid interpolation
	sample_points_accessible = sample_points_fractional[dgrid_interpolator(sample_points_fractional) > probe_radius]

	# Get the radii for the accessible sampling points
	sample_point_radii = dgrid_interpolator(sample_points_accessible) - probe_radius

	# Print statistics of the sample point radii
	print('Statistics on sphere sizes at sampling points in the void space')
	print('Sample point radii =', sample_point_radii)
	print_stat(sample_point_radii)

	print('==================================================')
	def points_to_nodes_distances(sample_points_block, nodes_fractional, cell=cell):
		"""
		Calculate distances from sample points to Voronoi nodes and determine valid radii.
		This function will be applied to each block of sample points, to be parallelized using Dask.
		
		Parameters:
		- sample_points_block (ndarray): Block of sample points in Cartesian coordinates.
		- nodes_fractional (ndarray): Fractional coordinates of the Voronoi nodes.
		- cell (ndarray): The cell matrix for coordinate transformation.

		Returns:
		- result (ndarray): An array of valid radii for the sample points.
		"""
		# Calculate the difference between sample points and nodes
		diff = np.expand_dims(sample_points_block, axis=1) - np.expand_dims(nodes_fractional, axis=0)
		diff = (diff - np.round(diff)).reshape(-1, 3)  # Wrap around periodic boundaries
		diff = np.dot(cell, diff.T).T  # Transform to Cartesian coordinates
		diff = np.linalg.norm(diff, axis=1).reshape(-1, nodes_fractional.shape[0])  # Compute distances

		# Print statistics of the distances
		# print_stat(diff)

		# Create a mask for valid node radii
		mask = (node_radii - probe_radius)> diff
		valid_radii = np.where(mask, node_radii - probe_radius, -np.inf)  # Replace invalid radii with -inf
		max_valid_radii = np.max(valid_radii, axis=1)  # Get the maximum valid radii

		# Print statistics of the maximum valid radii
		# print_stat(max_valid_radii)

		# Get the radii for the sample block
		sample_block_radii = dgrid_interpolator(sample_points_block) - probe_radius

		# Combine sample block radii and maximum valid radii
		result = np.array([s if np.logical_and(m != -np.inf, s > m) else m for s, m in zip(sample_block_radii, max_valid_radii)])

		return result * 2  # Return the result scaled by a factor of 2 converting radii to diameters

	# Create a Dask array from the accessible sample points
	sample_points_da = da.from_array(sample_points_accessible, chunks=(blocksize, 3))

	# Map the distance calculation function over the Dask array
	pore_sizes = sample_points_da.map_blocks(
		points_to_nodes_distances,
		nodes_fractional=nodes_fractional,
		cell=cell,
		chunks=(blocksize,),
		dtype='float32',
		drop_axis=1
	).compute()  # Compute the result
	print('For trustworthy sampled points')
	# Print statistics of the computed pore sizes
	print_stat(pore_sizes[np.isfinite(pore_sizes)])  # Only print finite values
	print('==================================================')
	return pore_sizes  # Return the computed pore sizes

## Run the function

In [37]:
print(path_to_cif.name)
atoms = read(path_to_cif)
dgrid = mgr.dgrid_from_atoms_cpu(atoms, spacing=0.25)
print(dgrid.dtype)
pore_sizes = compute_pore_sizes_for_psd(atoms, dgrid, probe_radius=1.25)

ABAVIJ_clean.cif


c:\Users\arung\miniforge3\envs\magic\Lib\site-packages\ase\io\cif.py:408: UserWarning:

crystal system 'triclinic' is not interpreted for space group Spacegroup(1, setting=1). This may result in wrong setting!



float64
Statistics on sphere sizes at Voronoi nodes 
(-ve values mean inside framework atoms)
shape: (3072,)
Max: 1.8992260176398106
Min: -0.49504540638659544
Mean: 0.46793441752350245
Variance: 0.38746569526471375
Statistics on sphere sizes at all sampling points 
(-ve values mean inside framework atoms)
shape: (50000,)
Max: 2.05610912742617
Min: -1.8271692441615837
Mean: -0.13165366468459683
Variance: 0.40916080595714593
Statistics on sphere sizes at sampling points in the void space
Sample point radii = [0.24786924 0.1848177  0.22075531 ... 0.07764527 0.05260158 0.18754554]
shape: (1442,)
Max: 0.8061091274261698
Min: 0.0003764995377977254
Mean: 0.2521934352230376
Variance: 0.03442264171178264
For trustworthy sampled points
shape: (472,)
Max: 1.6122182548523396
Min: 0.25077807868289614
Mean: 1.0952351056015386
Variance: 0.050897148037886274


## Plot the results

In [38]:
import numpy as np
import plotly.graph_objects as go

def get_pore_size_distribution_trace(distances, normalize=False, cumulative=False):
    # if normalize:
        # distances = distances / np.max(distances)
    
    # Create histogram data
    hist, bin_edges = np.histogram(distances, bins=np.linspace(0,100, 200), density=normalize)
    
    # Calculate cumulative distribution if required
    if cumulative:
        hist = np.cumsum(hist)
    binC = (bin_edges[1:] + bin_edges[:-1])/2
    # Create the histogram trace
    trace = go.Scatter(
        x=binC, 
        y=hist, 
        mode='lines', 
        name='Pore Size Distribution',
        line=dict(width=2)
    )
    
    return trace

# Get the histogram trace for distances_to_nodes
histogram_trace = get_pore_size_distribution_trace(pore_sizes, normalize=False, cumulative=False)

fig = go.Figure()
fig.add_trace(histogram_trace)
fig.update_layout(title='Pore Size Distribution',
                  xaxis_title='Pore Dia (\u212b)',
                  yaxis_title='Density',
                  barmode='overlay')
fig.show()



In [34]:
# frame_trace = mgr.plot_framework_as_mesh(atoms, stylize=True)
# cell_trace = mgr.plot_cell(atoms)
# fig = go.Figure(data= [*frame_trace, cell_trace], layout=mgr.dark_layout)
# fig.show()